# Kaggle submission notebook

## 1. Notebook setup
### 1.1. Imports

In [1]:
# Standard library
import json

# Third party
import pandas as pd

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.impute import KNNImputer
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import OneHotEncoder, LabelEncoder

### 1.2. Run configuration

In [2]:
SAMPLE = 0.01      # Fraction of data to use for training

IMPUTE = True      # True = do imputation, False = load imputation results from disk
KNN_NEIGHBORS = 3  # Number of neighbors to use for KNN imputation

### 1.3. Data loading

#### 1.3.1. Training data

In [3]:
train_df = pd.read_csv('../data/train.csv')
train_df = train_df.sample(frac=SAMPLE)
train_df.head()

,id,health_condition,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
595234,595234,at-risk,5.44,73.3,25.55,2178.0,7888.0,55.1,2.25,veg,medium,poor,active,NaN,female
95734,95734,at-risk,7.52,77.6,18.38,2396.0,9420.0,48.9,2.10,veg,medium,good,moderate,occasional,male
232334,232334,at-risk,5.10,67.2,23.10,2670.0,13259.0,60.6,NaN,balanced,medium,poor,active,occasional,female
655639,655639,unhealthy,5.66,80.6,24.17,2650.0,12367.0,33.2,3.20,balanced,high,poor,moderate,no,NaN
7187,7187,at-risk,NaN,86.9,25.57,2111.0,1294.0,40.1,2.04,balanced,NaN,good,sedentary,no,other


#### 1.3.2. Testing data

In [4]:
test_df = pd.read_csv('../data/test.csv')
test_df.head()

,id,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
0,690088,5.35,64.9,23.48,2745.0,14167.0,59.5,1.86,veg,high,poor,active,occasional,male
1,690089,NaN,83.1,22.42,1773.0,6801.0,24.5,2.40,balanced,high,poor,sedentary,yes,other
2,690090,6.68,59.7,24.14,3040.0,13250.0,48.5,2.76,balanced,medium,poor,active,no,NaN
3,690091,7.13,78.5,26.26,2494.0,6331.0,56.9,2.34,veg,low,good,moderate,yes,other
4,690092,5.49,77.7,23.29,1828.0,13894.0,39.4,2.45,veg,high,average,active,occasional,other


#### 1.3.3. Dataset metadata

In [5]:
with open("../data/schema.json", "r") as f:
    metadata = json.load(f)

features = metadata['features']
categorical_features = metadata['categorical_features']
continuous_features = metadata['continuous_features']
high_cardinality_feature = metadata['high_cardinality_feature']
label = metadata['label']

print(f'Categorical features: {" ,".join(categorical_features)}')
print(f'Continuous features: {", ".join(continuous_features)}')
print(f'High cardinality feature: {high_cardinality_feature}\n')
print(f'Label: {label}')

Categorical features: diet_type ,stress_level ,sleep_quality ,physical_activity_level ,smoking_alcohol ,gender
Continuous features: sleep_duration, heart_rate, bmi, calorie_expenditure, step_count, exercise_duration, water_intake
High cardinality feature: step_count

Label: health_condition


## 2. Data preprocessing

### 2.1. Feature encoding

In [6]:
# Scikit-learn one-hot encoder
feature_encoder = OneHotEncoder(sparse_output=False, drop='first')

# Adapt to training data categorical features
feature_encoder.fit(train_df[categorical_features])

# Transform training and testing data
encoded_train_features = feature_encoder.transform(train_df[categorical_features])
encoded_test_features = feature_encoder.transform(test_df[categorical_features])

# Convert encoded features to dataframe
encoded_train_df = pd.DataFrame(encoded_train_features, columns=feature_encoder.get_feature_names_out())
encoded_test_df = pd.DataFrame(encoded_test_features, columns=feature_encoder.get_feature_names_out())

# Replace original categorical features with encoded features
train_df = pd.concat([train_df.drop(columns=categorical_features), encoded_train_df], axis=1)
test_df = pd.concat([test_df.drop(columns=categorical_features), encoded_test_df], axis=1)

train_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 13736 entries, 595234 to 6900
Data columns (total 27 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   id                                 6901 non-null   float64
 1   health_condition                   6901 non-null   object 
 2   sleep_duration                     6152 non-null   float64
 3   heart_rate                         6831 non-null   float64
 4   bmi                                6766 non-null   float64
 5   calorie_expenditure                6340 non-null   float64
 6   step_count                         6764 non-null   float64
 7   exercise_duration                  6838 non-null   float64
 8   water_intake                       6473 non-null   float64
 9   diet_type_non-veg                  6901 non-null   float64
 10  diet_type_veg                      6901 non-null   float64
 11  diet_type_nan                      6901 non-null   floa

### 2.2. Label encoding

In [ ]:
# Scikit-learn label encoder
label_encoder = LabelEncoder()

# Convert string training labels to integer representation
train_df[label] = label_encoder.fit_transform(train_df[label])

: 

### 2.3. Imputation

In [ ]:
if IMPUTE: # Do the imputation

    # Distance weighted Scikit-learn KNN imputer
    imputer = KNNImputer(n_neighbors=KNN_NEIGHBORS, weights='distance', add_indicator=True)

    # Fit on training data, transform training and test data
    train_df = imputer.fit_transform(train_df)
    test_df = imputer.transform(test_df)

    # Save expensive imputation result
    train_df.to_csv("../data/train_imputed.csv", index=False)
    test_df.to_csv("../data/test_imputed.csv", index=False)

else: # Load prior imputation results

    train_df = pd.read_csv("../data/train_imputed.csv")
    test_df = pd.read_csv("../data/test_imputed.csv")

train_df.info()

**Note:** imputation of the full dataset takes about 4 hours.

## 3. Gradient boosting classifier

In [ ]:
# Scikit-learn gradient boosting classifier with defaults
gbc = GradientBoostingClassifier()

# Use cross validation to evaluate the model
scores = cross_val_score(
    gbc,
    train_df.drop(columns=['id', 'health_condition']),
    train_df['health_condition'], 
    cv=10,
    scoring='balanced_accuracy_score',
)

# Get 95% confidence interval bounds
upper_bound = scores.mean() + 1.96 * scores.std() / (len(scores) ** 0.5)
lower_bound = scores.mean() - 1.96 * scores.std() / (len(scores) ** 0.5)

# Show results
print(f"Cross-validation R2 scores: {''.join([f'{x:.3f}, ' for x in scores])}\n")
print(f"Mean cross-validation R2 score: {scores.mean():.3f}")
print(f"95% confidence interval for the R2 score: ({lower_bound:.3f}, {upper_bound:.3f})")

In [ ]:
# Fit the model on the entire training set
gbc.fit(train_df.drop(columns=['id', 'health_condition']), train_df['health_condition'])

# Make predictions on the test set
predictions = gbc.predict(test_df.drop(columns=['id']))

# Un-encode predictions to get string labels back
predictions = label_encoder.inverse_transform(predictions)

# Convert to dataframe
submission_df = pd.DataFrame({'id': test_df['id'], 'health_condition': predictions})

In [ ]:
# Save predictions to a CSV file
submission_df.to_csv('../data/submission.csv', index=False)